In [1]:
import sys
print(sys.executable)


C:\Users\CAdeK\anaconda3\envs\opentnsim\python.exe


In [2]:
# package(s) used for creating and geo-locating the graph
import networkx as nx
from pyproj import Geod
from shapely.geometry import Point, LineString
from opentnsim import graph

# package(s) related to the hydrodynamics and tidal windows
import xarray as xr
from opentnsim.tidal_accessibility import HasDraughtRestrictions
from opentnsim.core import Identifiable, Movable 
from opentnsim.graph import HasMultiDiGraph
from opentnsim.core.vessel_properties import VesselProperties
from opentnsim.port.port import IsPort, HasPortAccess
from opentnsim.port.terminals import IsTerminal, IsJetty, IsQuay, HasTerminal
from opentnsim.port.anchorages import IsAnchorage, PassesAnchorage
from opentnsim.vessel_traffic_service import VesselTrafficService
from opentnsim import tidal_accessibility
from opentnsim.tidal_window_constructor import vessel_specifications, \
                                               vertical_tidal_window_specifications, \
                                               vertical_tidal_window_input, \
                                               horizontal_tidal_window_specifications, \
                                               horizontal_tidal_window_method, \
                                               horizontal_tidal_window_input, \
                                               tidal_period, \
                                               accessibility, \
                                               NetworkProperties
from opentnsim.rule_constructor import vessel_characteristics, vessel_direction, vessel_type

# package(s) related to the simulation (creating the vessel, running the simulation)
import datetime
import simpy
import opentnsim

# package(s) needed for inspecting the output
import pandas as pd
import numpy as np

# package(s) for plotting
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

print("This notebook is executed with OpenTNSim version {}".format(opentnsim.__version__))

ModuleNotFoundError: No module named 'opentnsim.tidal_accessibility'

In [ ]:
%load_ext autoreload
%autoreload 2

#### Define input
vessel

In [ ]:
vessel_length = 200
vessel_beam = 40
vessel_draught = 12
vessel_speed = 4.5 #m/s
vessel_loading_time = 6 #hours

infrastructure

In [ ]:
waterway_length = 20000 #meters
waterway_depth = 15 #meters (NGD: nautical guaranteed depth = MBL: maintained bed level)

tide

In [ ]:
tidal_period = 12.5 #hours
tidal_range_sea = 2.0 #meters
tidal_propagation_speed = np.sqrt(9.81*waterway_depth) #meters per second (np.sqrt(grav_acc*waterway_depth)), grav_acc=9.81
tidal_amplication = 0.8 #amplification factor

tidal restrictions

In [ ]:
#percentage of the draught as UKC (10% of draught is common)
ukc_percentage_sea = 10.0 #% of draught 
ukc_percentage_canal = 10.0 #% of draught

#static value as UKC (0.5 m is common, but not in combination with ukc_percentage)
ukc_static_sea = 0.0 #meters
ukc_static_canal = 0.0 #meters

#freshwater allowance as percentage of draught as additional UKC: saltwater at sea (no sinkage), fresher water in port (5% extra sinkage common)
fwa_sea = 0.0 #% of draught
fwa_canal = 0.0 #% of draught

#### 0. Create environment

In [ ]:
# set simulation time
simulation_start = datetime.datetime(2024, 1, 1, 0, 0, 0)
simulation_stop = datetime.datetime(2024, 1, 8, 0, 0, 0)

# create time series aligned with simulation time
timestep = pd.Timedelta(minutes=5)
time_series = pd.date_range(simulation_start,simulation_stop,freq=timestep)

env = simpy.Environment(initial_time=simulation_start.timestamp())
env.simulation_start = simulation_start
env.simulation_stop = simulation_stop
env.epoch = simulation_start

#### 1. Create graph
Next we create a network (a graph) along which the vessel can move. For this case we create a single edge of 100 km exactly.

In [ ]:
# initialize geodetic calculator with WGS84 ellipsoid
geod = Geod(ellps="WGS84")

In [ ]:
# starting point (longitude, latitude)
coordinates_sea = (0,0)

# compute the other point 100 km East from Point 0
coordinates_port = geod.fwd(coordinates_sea[0], coordinates_sea[1], 90 ,waterway_length) # East from Point 0

# define nodes with their geographic coordinates
coordinates_waterway = {"Sea": (coordinates_sea[0], coordinates_sea[1]),
                        "Port basin": (coordinates_port[0], coordinates_port[1]),}

In [ ]:
# # Test met multiple nodes net zoals in 0301 1/3

# # starting point (longitude, latitude)
# coordinates_sea = (0, 0)

# # number of nodes along the route (incl. Sea and Port basin)
# number_of_nodes = 11  # bv. 2 = oud, 11 = multi-node

# coordinates_waterway = {}
# coordinates_waterway[0] = coordinates_sea
# coordinates_previous_node = coordinates_sea
# for node in range(number_of_nodes - 1):
#     coordinates_node = geod.fwd(
#         coordinates_previous_node[0],
#         coordinates_previous_node[1],
#         90, waterway_length / (number_of_nodes - 1)
#     )#east from previous node
#     coordinates_waterway[node + 1] = (coordinates_node[0], coordinates_node[1])
#     coordinates_previous_node = coordinates_node



In [ ]:
# # Test met multiple nodes net zoals in 0301 2/3

# # create list of edges
# edges = []
# for node_start,node_stop in zip(range(0,number_of_nodes-1),range(1,number_of_nodes)):
#     edges.append((node_start,node_stop))
#     edges.append((node_stop,node_start))

In [ ]:
# create list of edges
edges = [("Sea", "Port basin"), ("Port basin", "Sea")] # bi-directional edge

In [ ]:
# create a directed graph
FG = nx.DiGraph()

# add nodes
for name, coordinate in coordinates_waterway.items():
    FG.add_node(name, geometry=Point(coordinate[0], coordinate[1]))

# add edges
for edge in edges:
    FG.add_edge(edge[0], edge[1], geometry=LineString([Point(coordinates_sea),Point(coordinates_port[:-1])]), weight=1, length_m=waterway_length)

In [ ]:
# # Test met multiple nodes net zoals in 0301 3/3
# # create a directed graph
# FG = nx.DiGraph()

# # add nodes
# for name, coordinate in coordinates_waterway.items():
#     FG.add_node(name, geometry=Point(coordinate[0], coordinate[1]))

# # add edges
# for edge in edges:
#     waterway_geometry = LineString([Pointve(coordinates_waterway[edge[0]]),Point(coordinates_waterway[edge[1]])])
#     FG.add_edge(edge[0], edge[1], geometry=waterway_geometry, weight=1, length_m=waterway_length/(number_of_nodes-1))

In [ ]:
opentnsim.graph.plot_graph(FG)

In [ ]:
# add graph to environment
env.graph = FG

# voor test
for n in env.graph.nodes:
    env.graph.nodes[n]["depth"] = waterway_depth

#### 1a. Adding a Vessel Traffic Service (VTS)

In [ ]:
# create empty hydrodynamic dataset
hydrodynamic_data = xr.Dataset()

In [ ]:
# # test met multiple nodes net als 0301 1/3

# water_level_data = np.zeros((number_of_nodes,len(time_series)))
# for node in FG.nodes:
#     route_to_sea = nx.dijkstra_path(FG,0,node)
#     distance_to_sea = 0.
#     for node_start,node_stop in zip(route_to_sea[:-1],route_to_sea[1:]):
#         distance_to_sea += FG.edges[node_start,node_stop]['length_m']

#     amplification_factor = 1 - 0.2*(distance_to_sea/waterway_length)
#     phase_shift = pd.Timedelta(seconds=distance_to_sea/tidal_propagation_speed)
#     tidal_amplitude = amplification_factor*(tidal_range_sea/2)
#     tidal_water_level = tidal_amplitude*np.sin(2*np.pi*((time_series-simulation_start-phase_shift)/(12.5*pd.Timedelta(hours=1))))
#     water_level_data[node] = tidal_water_level
# water_level_data_array = xr.DataArray(data=water_level_data,coords={'STATION':FG.nodes,'TIME':time_series})

In [ ]:
# # test met multiple nodes net als 0301 2/3
# MBL_data = waterway_depth*np.ones([number_of_nodes,len(time_series)])
# MBL_data_array = xr.DataArray(data=MBL_data,coords={'STATION':FG.nodes,'TIME':time_series})
# hydrodynamic_data['Water level'] = water_level_data_array
# hydrodynamic_data['MBL'] = MBL_data_array


In [ ]:
tidal_phase_shift = pd.Timedelta(seconds=waterway_length/tidal_propagation_speed)

tidal_water_level_sea = (tidal_range_sea/2)*np.sin(2*np.pi*((time_series-simulation_start)/(12.5*pd.Timedelta(hours=1))))
tidal_water_level_port = tidal_amplication*(tidal_range_sea/2)*np.sin(2*np.pi*((time_series-simulation_start-tidal_phase_shift)/(12.5*pd.Timedelta(hours=1))))

In [ ]:
water_level_data = xr.DataArray(data=[tidal_water_level_sea,tidal_water_level_port],coords={'STATION':FG.nodes,'TIME':time_series})
MBLs = waterway_depth*np.ones(len(time_series))
MBL_data = xr.DataArray(data=[MBLs,MBLs],coords={'STATION':FG.nodes,'TIME':time_series})
hydrodynamic_data['Water level'] = water_level_data
hydrodynamic_data['MBL'] = MBL_data

In [ ]:
# # test met multiple nodes net als 0301 3/3

# fig,ax = plt.subplots(figsize=[12,3])
# ax.set_facecolor('slategray')
# pd.DataFrame(hydrodynamic_data['Water level'].T,index=hydrodynamic_data['TIME']).plot(ax=ax,cmap='Blues_r')
# ax.set_xlabel('Time')
# ax.set_ylabel('Water level [m]')
# ax.set_xlim(time_series[0],time_series[-1])
# #ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
# ax.legend(loc='upper right',frameon=True,bbox_to_anchor=[1.1,1.0375],facecolor='slategray',edgecolor='k');

In [ ]:
fig,ax = plt.subplots(figsize=[12,3])
ax.set_facecolor('slategrey')
cmap = plt.get_cmap('Blues_r')
ax.plot(time_series,tidal_water_level_sea,color=cmap(0),label='sea')
ax.plot(time_series,tidal_water_level_port,color=cmap(cmap.N),label='port')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
ax.set_xlabel('Time')
ax.set_ylabel('Water level [m]')
ax.set_xlim(time_series[0],time_series[-1])
ax.legend(loc='upper right',frameon=True,bbox_to_anchor=[1.1,1.0375],facecolor='slategray',edgecolor='k');

In [ ]:
vessel_traffic_service = VesselTrafficService(hydrodynamic_data = hydrodynamic_data)
env.vessel_traffic_service = vessel_traffic_service

In [ ]:
ukc_p = [ukc_percentage_sea/100,ukc_percentage_canal/100] #percentage of the UKC (10% of draught is common)
ukc_s = [ukc_static_sea,ukc_static_canal] #static value of the UKC
fwa = [fwa_sea/100,fwa_canal/100] #freshwater allowance as percentage of UKC: saltwater at sea (no sinkage), fresher water in port (5% extra sinkage)

network_properties = NetworkProperties()
for index,node in enumerate(FG.nodes):
    FG.nodes[node]['Vertical tidal restriction'] = {}
    vertical_tidal_window_inputs = []

    vessel_specification = vessel_specifications({vessel_characteristics.min_ge_Draught: 0.0},'x',vessel_direction.inbound)
    window_specification = vertical_tidal_window_specifications(ukc_s = ukc_s[index],
                                                                ukc_p = ukc_p[index],
                                                                fwa = fwa[index],)
    
    vertical_tidal_window_inputs.append(vertical_tidal_window_input(vessel_specifications = vessel_specification,
                                                                    window_specifications = window_specification))

    network_properties.append_vertical_tidal_restriction_to_network(env.graph,node,vertical_tidal_window_inputs)

In [ ]:
# # test new tidal restrictions

# ukc_percentage = 10.0 * np.ones(number_of_nodes)
# ukc_static     = 0.0  * np.ones(number_of_nodes)
# fwa            = 0.0  * np.ones(number_of_nodes)

# ukc_p = ukc_percentage / 100
# ukc_s = ukc_static
# fwa   = fwa / 100

#### 1b. Adding Port Infrastructure

In [ ]:
port = IsPort(env=env, name = "Port", port_entry_nodes=["Sea"])
# node 0 = origineel 'Sea'maar nu node 0

In [ ]:
anchorage = IsAnchorage(env=env, capacity=20, name = 'Anchorage', node = 'Sea', geometry=env.graph.nodes['Sea']['geometry'], port = port)
# node 0 = origineel 'Sea'maar nu node 0

In [ ]:
harbour_basin_depth = waterway_depth + tidal_amplication*(tidal_range_sea/2) + 0.5

quay = opentnsim.port.terminals.IsQuay(env=env, length=500, depth=harbour_basin_depth, name='Quay',node='Port basin')
jetty = opentnsim.port.terminals.IsJetty(env=env,length=250, depth=harbour_basin_depth, name='Jetty',node='Port basin')
berths = [quay,jetty]

terminal = IsTerminal(env=env, name = 'Terminal', berths=berths, port = port)

# node 0 = origineel 'Sea'maar nu node 0, en node 10 = origineel 'Port basin'

#### 2. Create agents

In [ ]:
# make your preferred Vessel class out of available mix-ins.
Vessel = type(
    "Vessel", 
    (HasPortAccess,
     HasTerminal,
     PassesAnchorage,
     HasDraughtRestrictions,
     HasMultiDiGraph,
     Identifiable, # allows to give the object a name and a random ID,
     Movable,      # allows the object to move, with a fixed speed, while logging this activity
     VesselProperties,
    ), 
    {}
)

In [ ]:
def mission(env, vessel, arrival_time = simulation_start):
    """
    Method that defines the mission of the vessel. 
    
    In this case: 
        keep moving along the path until its end point is reached
    """
    if arrival_time < simulation_start:
        raise ValueError("The vessel's arrival time at the port should be later that the simulation start time")
    else:
        delay = (arrival_time - simulation_start).total_seconds()
        yield vessel.env.timeout(delay)
    yield from vessel.move()

In [ ]:
def create_vessel(name, route, next_destination, 
                  vessel_speed = 4.5, berthing_time = 15, loading_time = 6, deberthing_time = 5, 
                  vessel_length = vessel_length, vessel_beam = vessel_beam, vessel_draught = vessel_draught):
    data_vessel = {"env": env,                                        # needed for simpy simulation
                   "name": name,                                      # required by Identifiable
                   "geometry": env.graph.nodes[route[0]]['geometry'], # required by Locatable
                   "route": route,                                    # required by Routeable
                   "v": vessel_speed,                                 # required by Movable, 1 m/s to check if the distance is covered in the expected time
                   "bound": 'inbound',
                   "terminal": terminal,
                   "berthing_time":berthing_time, #minutes
                   "loading_time": loading_time, #hours
                   "deberthing_time":deberthing_time, #minutes
                   "next_destination":next_destination,
                   "type":"tanker",
                   "L":vessel_length,
                   "B":vessel_beam,
                   "T":13.5,
                   }
    vessel = Vessel(**data_vessel)
    return vessel

In [ ]:
# create vessel from a dict 
route = nx.dijkstra_path(env.graph, "Sea", 'Port basin')
vessel_1 = create_vessel("vessel_1", route, "Sea")
vessel_2 = create_vessel("vessel_2", route, "Sea")
vessel_3 = create_vessel("vessel_3", route, "Sea")
vessel_4 = create_vessel("vessel_4", route, "Sea")

# 0 ="Sea"; 10 = "Port Basin"

# start the simulation
vessel_1.mission = env.process(mission(env, vessel_1))
vessel_2.mission = env.process(mission(env, vessel_2, arrival_time = simulation_start + pd.Timedelta(hours=3)))
vessel_3.mission = env.process(mission(env, vessel_3, arrival_time = simulation_start + pd.Timedelta(hours=4)))
vessel_4.mission = env.process(mission(env, vessel_4, arrival_time = simulation_start + pd.Timedelta(hours=5)))


env.vessels = [vessel_1, vessel_2, vessel_3, vessel_4]



In [ ]:
def quay_has_space(quay, vessel):
    df = quay.availability_quay_positions
    return ((df["Length_available"] >= vessel.L) &
            (df["Occupant"].isna() | (df["Occupant"] == None))).any()


def jetty_has_space(jetty):
    # jetty gebruikt een SimPy resource
    return jetty.resource.count < jetty.resource.capacity


In [ ]:
def choose_berth(env, vessel, quay, jetty, check_interval_s=60):
    """
    Kies quay of jetty als er plek is, anders wachten.
    """
    while True:
        if quay_has_space(quay, vessel):
            vessel.berth = quay
            vessel.logbook.append(("Using Quay", env.now, vessel.distance, None))
            return

        if jetty_has_space(jetty):
            vessel.berth = jetty
            vessel.logbook.append(("Using Jetty", env.now, vessel.distance, None))
            return

        # geen plek -> wachten
        vessel.logbook.append(("Waiting for Terminal start", env.now, vessel.distance, None))
        yield env.timeout(check_interval_s)
        vessel.logbook.append(("Waiting for Terminal stop", env.now, vessel.distance, None))


#### 3. Run simulation

In [ ]:
# # --- add alias node 'Sea' pointing to node 0 ---
# if "Sea" not in env.graph.nodes:
#     env.graph.add_node("Sea", **env.graph.nodes[0])  # copy node attributes (geometry etc.)
#     env.graph.add_edge("Sea", 0, length_m=0, weight=0)
#     env.graph.add_edge(0, "Sea", length_m=0, weight=0)


In [ ]:
env.run()

In [ ]:
## hier zit nu de waiting for tide en congestie voor de terminals in (denk ook met de Berth en de jetty; moet nog gecontroleerd worden)
## moet gecontroleerd worden hoe hier de tidal window wordt berekend. En welke tidal restrictions hier aan gebonden zijn.
## Dan moet er ook gecontroleerd worden of de waiting voor de tide, die overgaat op de waiting for berth samen kloppen. Welke is leidend in dit figuurtej. moest zonder de tide vessel 4 ook wachten tot 08.00 of niet. (uit de berekeningn zou ook komen dat de tidal window restrictie al ron 05.00 al weer klaar is) 
## het zou goed zijn om ook het plotje van de tidal windows uit notebook 0301 te maken. EN ook van de berth capacity wat hier uitkomt. 

## En misschien goed om voor een hele dag te plotten met schepen die ook later aankomen. 
# Is een gantt chart wel de juiste manier op het in beeld  te brengen? wil je niet liever getallen? en procenten 


#### 4. Inspect output

##### Logbook
We can now inspect  the simulation output by inspecting the _vessel.logbook_. Note that the _Log_ mix-in was included when we added _Movable_. The _vessel.logbook_ keeps track of the moving activities of the vessel. For each discrete event OpenTNSim logs an event message, the start/stop time and the location. The _vessel.logbook_ is of type dict. For convenient inspection it can be loaded into a Pandas dataframe. 

In [ ]:
# load the logbook data into a dataframe
for vessel in env.vessels:
    df = pd.DataFrame.from_dict(vessel.logbook)
    
    print("'{}' logbook data:".format(vessel.name))  
    print('')
    
    display(df)
    
    if not df.empty:
        trip_distance = opentnsim.graph.calculate_distance_along_path(FG, vessel.route)
        trip_duration = datetime.timedelta.total_seconds(vessel.logbook[-1]['Timestamp'] - vessel.logbook[0]['Timestamp'])
        print("'{}' travelled a distance of {:.1f} meters".format(vessel.name, trip_distance))
        print("'{}' took {:.1f} seconds to arrive at its destination".format(vessel.name, trip_duration)) 
        print("'{}' travelled at an average speed of {:.1f} meters per second".format(vessel.name, trip_distance/trip_duration))
    else:
        print("The port was not accessible for vessel")
    print('')
    print

    

Two

In [ ]:
fig = quay.plot_historic_quay_planning()
fig

# geen onderscheidt tussen quay en jetty 

In [ ]:
# gant chart van d e4 schepen en dan tide toevoegen? 
# test gant chart 


gantt_rows = []

for vessel in env.vessels:
    df = pd.DataFrame(vessel.logbook)

    # zorg dat Timestamp datetime is
    df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
    df = df.dropna(subset=["Timestamp"])

    # sorteer op tijd
    df = df.sort_values("Timestamp").reset_index(drop=True)

    open_events = {}

    for _, row in df.iterrows():
        msg = row["Message"]
        t = row["Timestamp"]

        if msg.endswith("start"):
            activity = msg.replace(" start", "")
            open_events[activity] = t

        if msg.endswith("stop"):
            activity = msg.replace(" stop", "")
            if activity in open_events:
                gantt_rows.append({
                    "Vessel": vessel.name,
                    "Activity": activity,
                    "Start": open_events[activity],
                    "Finish": t
                })
                del open_events[activity]

gantt_df = pd.DataFrame(gantt_rows)


In [ ]:
# test gant chart 
fig, ax = plt.subplots(figsize=(12, 5))

vessels = gantt_df["Vessel"].unique()
colors = {
    "Sailing from node Sea to node Port basin": "#1f77b4",
    "Sailing from node Port basin to node Sea": "#1f77b4",
    "Berthing": "#ff7f0e",
    "Loading": "#2ca02c",
    "Deberthing": "#d62728",
    "Waiting for Terminal": "#7f7f7f",
    "Waiting for Tide": "#9467bd",
}

for i, vessel in enumerate(vessels):
    vessel_data = gantt_df[gantt_df["Vessel"] == vessel]

    for _, row in vessel_data.iterrows():
        color = colors.get(row["Activity"], "#bbbbbb")

        ax.barh(
            vessel,
            (row["Finish"] - row["Start"]).total_seconds() / 3600,
            left=row["Start"],
            color=color,
            edgecolor="black"
        )
from matplotlib.patches import Patch

legend_elements = [
    Patch(facecolor=color, edgecolor="black", label=activity)
    for activity, color in colors.items()
]

ax.legend(
    handles=legend_elements,
    title="Activity",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

start_zoom = pd.Timestamp("2024-01-01 00:00:00")
end_zoom   = pd.Timestamp("2024-01-02 00:00:00")

ax.set_xlim(start_zoom, end_zoom)


ax.set_xlabel("Time")
ax.set_ylabel("Vessel")
ax.set_title("Gantt chart – vessel operations")
plt.tight_layout()
plt.show()

